# V4 vs. hybrid vs. selective fusion — mAP50 and mAP50–95

**Evaluation only, no training or model changes.**

Uses all **97 V4 validation images** with the original labels and **confidence 0.001**, IoU thresholds 0.50 to 0.95. The old V1 utensil specialist's original checkpoint is required. We compare three candidates:

- **V4**: original four-class detector.
- **Hybrid**: V4 food/plate/wrapper + original V1 utensil specialist.
- **Fusion (experimental)**: specialist utensils plus only V4 utensils with confidence ≥0.50; duplicate utensils suppressed at IoU ≥0.55.

**Important:** The validation annotations remain provisional and historically exposed. Treat these as development results, not independent test performance. Use a T4 GPU if available; CPU is possible but can be slow on the YOLO11x specialist.


## 1. Install and fetch the updated evaluator


In [ ]:
import subprocess,sys,torch,hashlib,zipfile,json
from pathlib import Path
from google.colab import files
subprocess.run([sys.executable,"-m","pip","install","-q","ultralytics==8.4.175"],check=True)
repo=Path("/content/yolo-cafeteria-research")
if repo.exists():
    subprocess.run(["git","-C",str(repo),"pull","--ff-only"],check=True)
else:
    subprocess.run(["git","clone","--depth","1","https://github.com/cx-57/yolo-cafeteria-research.git",str(repo)],check=True)
script=repo/"experiments/yolo-v4-4class/v4_hybrid_validation.py"
assert script.is_file(),"Evaluator missing from GitHub"
device="0" if torch.cuda.is_available() else "cpu"
print("Inference device:",device,flush=True)
def digest(raw): return hashlib.sha256(raw).hexdigest()


## 2. Upload the complete **V4 dataset ZIP** (151 MB)


In [ ]:
expected="YOLO_V4_865_4class_audit_pending.zip"
print("Upload:",expected,flush=True)
upload=files.upload()
assert expected in upload,"Wrong ZIP. Upload the V4 dataset archive."
archive=Path("/content")/expected
archive.write_bytes(upload[expected])
assert digest(upload[expected])=="95a386dbd9cf707e642237b92326a27ffd4958b893d1e0a7089e463b823c91e7","V4 ZIP hash mismatch"
root=Path("/content/yolo_v4_four_class")
if not (root/"source_manifest.json").exists():
    with zipfile.ZipFile(archive) as z:
        assert z.testzip() is None,"Corrupt archive"
        assert all(Path(m.filename).parts[:1]==("yolo_v4_four_class",) for m in z.infolist())
        z.extractall("/content")
assert (root/"source_manifest.json").is_file()
print("V4 dataset extracted, validation split has 97 photos",flush=True)


## 3. Upload the trained V4 results ZIP (**not** the dataset ZIP)


In [ ]:
print("Upload YOLO_V4_model_results.zip",flush=True)
upload=files.upload()
assert "YOLO_V4_model_results.zip" in upload,"Upload V4 trained results ZIP"
with zipfile.ZipFile(Path("/content")/"YOLO_V4_model_results.zip","w") as _:
    pass
# Use uploaded bytes directly, avoiding Colab's automatic '(1)' filename suffix
raw=upload["YOLO_V4_model_results.zip"]
from io import BytesIO
with zipfile.ZipFile(BytesIO(raw)) as z:
    weights=z.read("cafeteria/train/weights/best.pt")
assert digest(weights)=="0be9895e6c60bfe0b5fd478b01b6aef6484c24f827bf20ea7e65e7e34f66f343","Wrong V4 best.pt"
v4=Path("/content/v4_best.pt")
v4.write_bytes(weights)
print("V4 checkpoint verified",flush=True)


## 4. Upload the original V1 `utensil.pt` from your Mac


In [ ]:
print("Upload the frozen V1 weights/utensil.pt",flush=True)
upload=files.upload()
assert len(upload)==1,"Upload exactly one utensil.pt"
name,raw=next(iter(upload.items()))
assert name.lower().endswith(".pt"),"Expected utensil.pt"
assert digest(raw)=="ce4956547f0c421db1dc4b3d2fa7c28d9d9ae2fc44ddf5acf333ff211c3d4c7f","Wrong V1 specialist checkpoint"
ut=Path("/content/utensil_v1.pt")
ut.write_bytes(raw)
print("Specialist verified",flush=True)


## 5. Evaluate on 97 validation photos; download real AP measurements


In [ ]:
output=Path("/content/V4_hybrid_mAP_validation")
if output.exists():
    n=2
    while Path(str(output)+f"_run{n}").exists():n+=1
    output=Path(str(output)+f"_run{n}")
cmd=[sys.executable,"-u",str(script),"--root",str(root),
     "--v4",str(v4),"--utensil",str(ut),"--output",str(output),"--device",device]
print("Running mAP50 + mAP50–95 for V4, hybrid, fusion...",flush=True)
result=subprocess.run(cmd,check=True)
report=output/"hybrid_val_map.json"
print(json.dumps(json.loads(report.read_text())["results"],indent=2),flush=True)
files.download(str(report))
